In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from joblib import Parallel, delayed
from study_config import *
from forecast_eval import *

PROC = Path("../data/processed")
FIG = Path("../outputs/figures")
TAB = Path("../outputs/tables")

base = pd.read_csv(PROC / "forecast_results_paired.csv")
panel = pd.read_csv(PROC / "forecast_panel.csv")
panel["per"] = pd.PeriodIndex(panel["ym"], freq="M")
train_arrays = (panel[panel["role"] == "train"].sort_values("per")
                .groupby(KEYS)["log_price"].apply(lambda s: s.to_numpy()))

base_models = {"snaive": "f_snaive", "naive": "f_naive", "sarima": "f_sarima"}
sm = series_metrics(base, base_models)
print("Series-windows in the paired set:", base.groupby(KEYS).ngroups, "| training series:", len(train_arrays))

Series-windows in the paired set: 4162 | training series: 4163


In [2]:
sm_no = sm[sm["commodity"] != "Onions (red)"]
print(sm_no.groupby(["window", "model"]).size().unstack())
for a, b in [("naive", "snaive"), ("sarima", "snaive"), ("sarima", "naive")]:
    c = compare_gm(sm_no, a, b)
    print(f"\nWithout red onions: {a} vs {b} (MAPE)")
    display(c[c["metric"] == "MAPE"].drop(columns="metric"))
    print("Verdict (all three metrics):", {k[0]: v for k, v in overall(c).items()})
    c.to_csv(TAB / f"13_no_onions_{a}_vs_{b}.csv", index=False)

model          naive  sarima  snaive
window                              
Pandemic        1948    1948    1948
Post-pandemic   1949    1949    1949
Pre-pandemic     137     137     137

Without red onions: naive vs snaive (MAPE)


,window,group,series,ties_%,gm_ratio,ci,verdict
2,Pre-pandemic,all,137,1.5,0.589,0.496-0.679,naive better
5,Pandemic,all,1948,0.3,0.676,0.650-0.702,naive better
8,Post-pandemic,all,1949,0.4,0.747,0.725-0.771,naive better


Verdict (all three metrics): {'Pandemic': 'naive better', 'Post-pandemic': 'naive better', 'Pre-pandemic': 'naive better'}

Without red onions: sarima vs snaive (MAPE)


,window,group,series,ties_%,gm_ratio,ci,verdict
2,Pre-pandemic,all,137,0.7,0.622,0.541-0.704,sarima better
5,Pandemic,all,1948,0.3,0.718,0.693-0.743,sarima better
8,Post-pandemic,all,1949,0.2,0.753,0.735-0.773,sarima better


Verdict (all three metrics): {'Pandemic': 'sarima better', 'Post-pandemic': 'sarima better', 'Pre-pandemic': 'sarima better'}

Without red onions: sarima vs naive (MAPE)


,window,group,series,ties_%,gm_ratio,ci,verdict
2,Pre-pandemic,all,137,45.3,1.056,0.991-1.180,no clear winner
5,Pandemic,all,1948,52.1,1.061,1.044-1.079,naive better
8,Post-pandemic,all,1949,34.3,1.008,0.995-1.023,no clear winner


Verdict (all three metrics): {'Pandemic': 'naive better', 'Post-pandemic': 'no clear winner', 'Pre-pandemic': 'no clear winner'}


In [3]:
for a, b in [("naive", "snaive"), ("sarima", "naive")]:
    c = compare_gm(sm, a, b, by="category")
    print(f"\nBy category: {a} vs {b} (MAPE)")
    display(c[c["metric"] == "MAPE"].drop(columns="metric"))
    c.to_csv(TAB / f"13_by_category_{a}_vs_{b}.csv", index=False)


By category: naive vs snaive (MAPE)


,window,group,series,ties_%,gm_ratio,ci,verdict
6,Pre-pandemic,cereals and tubers,44,0.0,0.795,0.704-0.908,naive better
7,Pre-pandemic,"meat, fish and eggs",61,3.3,0.359,0.255-0.480,naive better
8,Pre-pandemic,vegetables and fruits,48,0.0,0.854,0.747-0.965,naive better
17,Pandemic,cereals and tubers,226,0.4,0.597,0.532-0.663,naive better
18,Pandemic,"meat, fish and eggs",748,0.1,0.609,0.577-0.643,naive better
19,Pandemic,pulses and nuts,86,2.3,0.689,0.616-0.773,naive better
20,Pandemic,vegetables and fruits,944,0.2,0.786,0.743-0.826,naive better
29,Post-pandemic,cereals and tubers,226,0.0,0.587,0.562-0.615,naive better
30,Post-pandemic,"meat, fish and eggs",748,0.3,0.665,0.625-0.705,naive better
31,Post-pandemic,pulses and nuts,86,4.7,0.724,0.656-0.791,naive better



By category: sarima vs naive (MAPE)


,window,group,series,ties_%,gm_ratio,ci,verdict
6,Pre-pandemic,cereals and tubers,44,36.4,1.047,1.009-1.096,naive better
7,Pre-pandemic,"meat, fish and eggs",61,50.8,1.118,0.974-1.393,no clear winner
8,Pre-pandemic,vegetables and fruits,48,41.7,0.997,0.964-1.032,no clear winner
17,Pandemic,cereals and tubers,226,50.0,1.089,1.034-1.172,naive better
18,Pandemic,"meat, fish and eggs",748,50.0,1.078,1.050-1.110,naive better
19,Pandemic,pulses and nuts,86,61.6,1.029,0.973-1.091,no clear winner
20,Pandemic,vegetables and fruits,944,53.3,1.040,1.022-1.059,naive better
29,Post-pandemic,cereals and tubers,226,54.9,1.024,1.004-1.047,naive better
30,Post-pandemic,"meat, fish and eggs",748,36.2,1.040,1.012-1.076,naive better
31,Post-pandemic,pulses and nuts,86,41.9,1.060,1.017-1.108,naive better


In [4]:
def make_fit(grid, trend="n", steps=12, max_jump=2.0):
    from statsmodels.tsa.arima.model import ARIMA
    def fit(y):
        best, spec = None, None
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            for p, q, P, Q in grid:
                try:
                    m = ARIMA(y, order=(p, 1, q), seasonal_order=(P, 0, Q, 12), trend=trend).fit()
                except Exception:
                    continue
                if np.isfinite(m.aic) and (best is None or m.aic < best.aic):
                    best, spec = m, (p, q, P, Q)
            if best is None:
                return None, None, "no model converged"
            fc = np.asarray(best.forecast(steps), dtype=float)
        if not np.isfinite(fc).all():
            return None, None, "non-finite forecast"
        if np.abs(fc - y[-1]).max() > max_jump:
            return None, None, "unstable forecast"
        return fc, spec, ""
    return fit

GRID12 = [(p, q, P, Q) for p in (0, 1) for q in (0, 1) for (P, Q) in ((0, 0), (1, 0), (0, 1))]
GRID27 = [(p, q, P, Q) for p in (0, 1, 2) for q in (0, 1, 2) for (P, Q) in ((0, 0), (1, 0), (0, 1))]

def run_one(key, y, fit):
    fc, spec, why = fit(y)
    return key, fc, spec, why

N_JOBS, BATCH = -2, 200
LIMIT = None          # use 20 for a quick trial first

def run_variant(tag, fit, windows=None, trim=None, limit=None):
    out = PROC / f"sarima_{tag}"; out.mkdir(exist_ok=True)
    for w in (windows or list(WINDOWS)):
        f = out / f"sarima_{w}.csv"
        done = set()
        if f.exists():
            prev = pd.read_csv(f); done = set(zip(prev["market_id"], prev["commodity_id"]))
        todo = [(k, y[-trim:] if trim else y) for k, y in train_arrays.items()
                if k[0] == w and (k[1], k[2]) not in done]
        if limit:
            todo = todo[:limit]
        for i in range(0, len(todo), BATCH):
            res = Parallel(n_jobs=N_JOBS)(delayed(run_one)(k, y, fit) for k, y in todo[i:i + BATCH])
            rows = []
            for k, fc, spec, why in res:
                r = {"window": k[0], "market_id": k[1], "commodity_id": k[2],
                     "spec": str(spec) if spec else "", "fail_reason": why}
                for s in range(12):
                    r[f"fc{s + 1}"] = fc[s] if fc is not None else np.nan
                rows.append(r)
            pd.DataFrame(rows).to_csv(f, mode="a", header=not f.exists(), index=False)
            print(f"{tag} | {w}: {min(i + BATCH, len(todo))}/{len(todo)}")

fit12, fit27, fit_drift = make_fit(GRID12), make_fit(GRID27), make_fit(GRID12, trend="t")
run_variant("post25", fit12, windows=["Post-pandemic"], trim=25, limit=LIMIT)   # same model, 25 months of training
run_variant("wide27", fit27, limit=LIMIT)                                        # wider grid
run_variant("drift12", fit_drift, limit=LIMIT)                                   # drift term (trend="t" with d=1)

post25 | Post-pandemic: 200/2005
post25 | Post-pandemic: 400/2005
post25 | Post-pandemic: 600/2005
post25 | Post-pandemic: 800/2005
post25 | Post-pandemic: 1000/2005
post25 | Post-pandemic: 1200/2005
post25 | Post-pandemic: 1400/2005
post25 | Post-pandemic: 1600/2005
post25 | Post-pandemic: 1800/2005
post25 | Post-pandemic: 2000/2005
post25 | Post-pandemic: 2005/2005
wide27 | Pre-pandemic: 153/153
wide27 | Pandemic: 200/2005
wide27 | Pandemic: 400/2005
wide27 | Pandemic: 600/2005
wide27 | Pandemic: 800/2005
wide27 | Pandemic: 1000/2005
wide27 | Pandemic: 1200/2005
wide27 | Pandemic: 1400/2005
wide27 | Pandemic: 1600/2005
wide27 | Pandemic: 1800/2005
wide27 | Pandemic: 2000/2005
wide27 | Pandemic: 2005/2005
wide27 | Post-pandemic: 200/2005
wide27 | Post-pandemic: 400/2005
wide27 | Post-pandemic: 600/2005
wide27 | Post-pandemic: 800/2005
wide27 | Post-pandemic: 1000/2005
wide27 | Post-pandemic: 1200/2005
wide27 | Post-pandemic: 1400/2005
wide27 | Post-pandemic: 1600/2005
wide27 | Post-pa

In [5]:
data = base.copy()
models = dict(base_models)
logs = {}
for tag in ["post25", "wide27", "drift12"]:
    sar, long = load_variant(PROC / f"sarima_{tag}")
    logs[tag] = sar
    data = data.merge(long.rename(columns={"f": f"f_{tag}"}), on=KEYS + ["h"], how="left")
    models[tag] = f"f_{tag}"
smv = series_metrics(data, models)

rows = []
for tag in ["post25", "wide27", "drift12"]:
    sar = logs[tag]
    good = sar[sar["fail_reason"] == ""]
    rw = good.assign(rw=good["spec"].eq("(0, 0, 0, 0)")).groupby("window")["rw"].mean().round(3).to_dict()
    fails = sar.assign(f=sar["fail_reason"] != "").groupby("window")["f"].mean().round(3).to_dict()
    for ref in ["naive", "snaive"]:
        c = compare_gm(smv, tag, ref)
        for (w, g), v in overall(c).items():
            m = c[(c["window"] == w) & (c["metric"] == "MAPE")].iloc[0]
            rows.append({"variant": tag, "vs": ref, "window": w, "series": m["series"], "ties_%": m["ties_%"],
                         "gm_ratio_MAPE": m["gm_ratio"], "ci": m["ci"], "verdict_all_metrics": v,
                         "random_walk_share": rw.get(w), "fail_rate": fails.get(w)})
summ = pd.DataFrame(rows)
display(summ)
summ.to_csv(TAB / "13_sarima_variants_summary.csv", index=False)

print("\nTraining-length test: 25-month SARIMA vs 62-month SARIMA, Post-pandemic window")
c = compare_gm(smv, "post25", "sarima", windows=["Post-pandemic"])
display(c)       # ratio > 1 means the shorter training window is WORSE
c.to_csv(TAB / "13_post25_vs_sarima62.csv", index=False)

,variant,vs,window,series,ties_%,gm_ratio_MAPE,ci,verdict_all_metrics,random_walk_share,fail_rate
0,post25,naive,Post-pandemic,2005,50.7,1.028,1.015-1.039,naive better,0.503,0.0
1,post25,snaive,Post-pandemic,2005,0.2,0.767,0.748-0.787,post25 better,0.503,0.0
2,wide27,naive,Pandemic,2004,43.1,1.064,1.047-1.084,naive better,0.424,0.0
3,wide27,naive,Post-pandemic,2005,23.3,0.996,0.982-1.011,no clear winner,0.225,0.0
4,wide27,naive,Pre-pandemic,153,30.7,1.062,0.987-1.187,no clear winner,0.294,0.0
5,wide27,snaive,Pandemic,2004,0.2,0.733,0.708-0.759,wide27 better,0.424,0.0
6,wide27,snaive,Post-pandemic,2005,0.1,0.743,0.724-0.763,wide27 better,0.225,0.0
7,wide27,snaive,Pre-pandemic,153,0.7,0.629,0.554-0.708,wide27 better,0.294,0.0
8,drift12,naive,Pandemic,2004,0.0,1.319,1.270-1.375,naive better,0.456,0.0
9,drift12,naive,Post-pandemic,2005,0.0,1.109,1.076-1.138,naive better,0.286,0.0



Training-length test: 25-month SARIMA vs 62-month SARIMA, Post-pandemic window


,window,group,metric,series,ties_%,gm_ratio,ci,verdict
0,Post-pandemic,all,MAE,2005,21.8,1.019,1.005-1.031,sarima better
1,Post-pandemic,all,RMSE,2005,21.1,1.018,1.006-1.030,sarima better
2,Post-pandemic,all,MAPE,2005,21.1,1.018,1.004-1.031,sarima better


In [6]:
d = pd.read_csv(PROC / "retail_clean_flagged.csv", parse_dates=["date"])
d["cm"] = d["date"].dt.month
d["p"] = d["period"].map({"Pre-pandemic": "pre", "Pandemic": "pan", "Post-pandemic": "post"})
ids = ["market_id", "commodity_id", "category", "commodity", "market", "admin1"]
cmm = d.groupby(ids + ["p", "cm"])["price"].mean().reset_index()     # mean price per calendar month, per period
MIN_CM = 6                                                           # >= 6 calendar months present in BOTH periods

def balanced(a, b):
    A = cmm[cmm["p"] == a].rename(columns={"price": "pa"})
    B = cmm[cmm["p"] == b].rename(columns={"price": "pb"})
    m = A.merge(B, on=ids + ["cm"])
    g = m.groupby(ids).agg(cal_months=("cm", "size"), p1=("pa", "mean"), p2=("pb", "mean")).reset_index()
    g = g[g["cal_months"] >= MIN_CM].copy()
    g["pct_chg"] = (g["p2"] / g["p1"] - 1) * 100
    return g

for k, (a, b) in {"pan_post": ("pan", "post"), "pre_pan": ("pre", "pan"), "pre_post": ("pre", "post")}.items():
    g = balanced(a, b)
    orig = pd.read_csv(TAB / f"10_matched_series_{k}.csv")
    print(f"{k}: balanced series {len(g)} | median {g['pct_chg'].median():.1f}% | share rising {(g['pct_chg'] > 0).mean() * 100:.1f}% "
          f"|| Step 10: series {len(orig)}, median {orig['pct_chg'].median():.1f}%")
    display(g.groupby("category").agg(series=("pct_chg", "size"), median_pct=("pct_chg", "median")).round(1))
    g.to_csv(TAB / f"13_calendar_balanced_{k}.csv", index=False)
    if k == "pan_post":
        mm = g.merge(orig[["market_id", "commodity_id", "pct_chg"]], on=["market_id", "commodity_id"], suffixes=("_bal", "_orig"))
        diff = (mm["pct_chg_bal"] - mm["pct_chg_orig"]).abs()
        print(f"Series-level agreement: correlation {mm['pct_chg_bal'].corr(mm['pct_chg_orig']):.3f} | "
              f"median abs difference {diff.median():.2f} pp | max {diff.max():.1f} pp")
        reg = g.groupby("admin1")["pct_chg"].median().round(1).sort_values()
        print("Lowest regions:", reg.head(2).to_dict(), "| highest:", reg.tail(2).to_dict())

pan_post: balanced series 2615 | median 18.8% | share rising 92.0% || Step 10: series 2615, median 18.8%


,series,median_pct
category,,
cereals and tubers,370,21.5
"meat, fish and eggs",915,16.1
pulses and nuts,110,11.3
vegetables and fruits,1220,22.7


Series-level agreement: correlation 0.995 | median abs difference 0.15 pp | max 31.0 pp
Lowest regions: {'Autonomous region in Muslim Mindanao': 12.5, 'Region VII': 16.0} | highest: {'Region VI': 21.3, 'Region X': 22.9}
pre_pan: balanced series 38 | median 41.1% | share rising 94.7% || Step 10: series 38, median 43.9%


,series,median_pct
category,,
cereals and tubers,9,36.7
"meat, fish and eggs",12,35.6
pulses and nuts,3,22.4
vegetables and fruits,14,83.6


pre_post: balanced series 36 | median 74.3% | share rising 100.0% || Step 10: series 36, median 73.4%


,series,median_pct
category,,
cereals and tubers,7,63.4
"meat, fish and eggs",12,56.2
pulses and nuts,3,37.5
vegetables and fruits,14,88.4
